# Análisis de Ventas y Rentabilidad - Superstore Sales Dataset

## Objetivo
Analizar qué categorías, subcategorías, regiones y segmentos de clientes están asociados a mejores o peores resultados comerciales, utilizando el dataset Superstore Sales.

## Herramientas
- Python
- Pandas (manejo y limpieza de datos)
- Plotly (visualizaciones interactivas)
- Jupyter Notebook

---
## 1. Importación de librerías y carga de datos

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Cargar el dataset
df = pd.read_csv('Superstore_Sales.csv')

# Vista inicial del dataset
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

---
## 2. Limpieza y preparación de datos

In [ ]:
# Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")

In [ ]:
# Verificar duplicados
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")

# Eliminar duplicados si existen
if duplicados > 0:
    df = df.drop_duplicates()
    print(f"Se eliminaron {duplicados} filas duplicadas.")
    print(f"Nuevo tamaño del dataset: {df.shape}")

In [ ]:
# Conversión de columnas de fecha a formato datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

print("Tipos de datos actualizados:")
print(df[['Order Date', 'Ship Date']].dtypes)

In [ ]:
# Creación de variables derivadas

# Año y mes de la orden
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Year-Month'] = df['Order Date'].dt.to_period('M').astype(str)

# Días de envío (diferencia entre Ship Date y Order Date)
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

print("Nuevas columnas creadas:")
print(df[['Order Date', 'Order Year', 'Order Month', 'Order Year-Month', 'Shipping Days']].head())

In [ ]:
# Resumen después de la limpieza
print(f"Dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nRango de fechas: {df['Order Date'].min().strftime('%d/%m/%Y')} - {df['Order Date'].max().strftime('%d/%m/%Y')}")
print(f"Años cubiertos: {sorted(df['Order Year'].unique())}")
print(f"\nCategorías: {df['Category'].unique().tolist()}")
print(f"Regiones: {df['Region'].unique().tolist()}")
print(f"Segmentos: {df['Segment'].unique().tolist()}")

---
## 3. Análisis Exploratorio y Visualizaciones Interactivas

A continuación se presentan visualizaciones interactivas creadas con Plotly para analizar las ventas desde diferentes perspectivas.

### 3.1 Análisis temporal de ventas

In [ ]:
# Ventas mensuales a lo largo del tiempo (Gráfico de líneas)
ventas_mensuales = df.groupby('Order Year-Month')['Sales'].sum().reset_index()
ventas_mensuales.columns = ['Periodo', 'Ventas']

fig1 = px.line(
    ventas_mensuales,
    x='Periodo',
    y='Ventas',
    title='Evolución mensual de ventas (2015-2018)',
    labels={'Ventas': 'Ventas (USD)', 'Periodo': 'Mes'},
    markers=True
)

fig1.update_layout(
    xaxis_tickangle=-45,
    template='plotly_white',
    hovermode='x unified'
)

fig1.show()

**Interpretación:** Este gráfico de líneas muestra la evolución de las ventas mensuales a lo largo del periodo analizado. Se pueden identificar patrones estacionales, con picos de ventas típicamente en los últimos meses del año (noviembre-diciembre), lo cual es consistente con temporadas de compras de fin de año. Además, se observa una tendencia general de crecimiento en las ventas.

### 3.2 Análisis de ventas por categoría y subcategoría

In [ ]:
# Ventas por categoría (Gráfico de barras)
ventas_categoria = df.groupby('Category')['Sales'].sum().reset_index()
ventas_categoria = ventas_categoria.sort_values('Sales', ascending=False)

fig2 = px.bar(
    ventas_categoria,
    x='Category',
    y='Sales',
    title='Ventas totales por categoría',
    labels={'Sales': 'Ventas (USD)', 'Category': 'Categoría'},
    color='Category',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig2.update_layout(template='plotly_white', showlegend=False)
fig2.show()

In [ ]:
# Ventas por subcategoría (Gráfico de barras horizontal)
ventas_subcategoria = df.groupby('Sub-Category')['Sales'].sum().reset_index()
ventas_subcategoria = ventas_subcategoria.sort_values('Sales', ascending=True)

fig3 = px.bar(
    ventas_subcategoria,
    x='Sales',
    y='Sub-Category',
    orientation='h',
    title='Ventas totales por subcategoría',
    labels={'Sales': 'Ventas (USD)', 'Sub-Category': 'Subcategoría'},
    color='Sales',
    color_continuous_scale='Viridis'
)

fig3.update_layout(template='plotly_white', height=500)
fig3.show()

**Interpretación:** El gráfico de barras por categoría muestra que **Technology** lidera en ventas totales, seguido de **Furniture** y **Office Supplies**. Al desglosar por subcategoría, se identifican los productos más y menos vendidos, permitiendo enfocar estrategias comerciales en las subcategorías con mayor potencial.

### 3.3 Análisis por región y segmento de clientes

In [ ]:
# Ventas por región (Gráfico de dispersión: ventas promedio vs cantidad de órdenes)
ventas_region = df.groupby('Region').agg(
    Ventas_Totales=('Sales', 'sum'),
    Ventas_Promedio=('Sales', 'mean'),
    Cantidad_Ordenes=('Order ID', 'nunique')
).reset_index()

fig4 = px.scatter(
    ventas_region,
    x='Cantidad_Ordenes',
    y='Ventas_Promedio',
    size='Ventas_Totales',
    color='Region',
    title='Relación entre cantidad de órdenes y venta promedio por región',
    labels={
        'Cantidad_Ordenes': 'Cantidad de órdenes únicas',
        'Ventas_Promedio': 'Venta promedio (USD)',
        'Ventas_Totales': 'Ventas totales'
    },
    size_max=60
)

fig4.update_layout(template='plotly_white')
fig4.show()

**Interpretación:** El gráfico de dispersión permite comparar las regiones según su cantidad de órdenes y su venta promedio. El tamaño de las burbujas representa las ventas totales. Se puede identificar qué regiones tienen mayor volumen de órdenes vs. cuáles tienen tickets de compra más altos, información clave para estrategias de expansión o fidelización.

In [ ]:
# Ventas por segmento de cliente y categoría (Gráfico de barras agrupadas)
ventas_segmento_cat = df.groupby(['Segment', 'Category'])['Sales'].sum().reset_index()

fig5 = px.bar(
    ventas_segmento_cat,
    x='Segment',
    y='Sales',
    color='Category',
    barmode='group',
    title='Ventas por segmento de cliente y categoría',
    labels={'Sales': 'Ventas (USD)', 'Segment': 'Segmento', 'Category': 'Categoría'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig5.update_layout(template='plotly_white')
fig5.show()

**Interpretación:** Se observa que el segmento **Consumer** es el que genera mayor volumen de ventas en todas las categorías, seguido de **Corporate** y **Home Office**. Esto indica que la mayor parte de los ingresos proviene de consumidores individuales, lo que puede orientar decisiones de marketing y servicio.

### 3.4 Análisis de tiempos de envío

In [ ]:
# Distribución de días de envío por modo de envío
fig6 = px.box(
    df,
    x='Ship Mode',
    y='Shipping Days',
    color='Ship Mode',
    title='Distribución de días de envío según modo de envío',
    labels={'Shipping Days': 'Días de envío', 'Ship Mode': 'Modo de envío'},
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig6.update_layout(template='plotly_white', showlegend=False)
fig6.show()

**Interpretación:** El diagrama de caja muestra la distribución de tiempos de envío para cada modo. Como es esperado, "Same Day" tiene los tiempos más cortos, mientras que "Standard Class" presenta los más largos. También se pueden identificar valores atípicos que podrían indicar problemas logísticos.

### 3.5 Análisis de ventas por estado (Top 10)

In [ ]:
# Top 10 estados por ventas
top_estados = df.groupby('State')['Sales'].sum().reset_index()
top_estados = top_estados.sort_values('Sales', ascending=False).head(10)

fig7 = px.bar(
    top_estados,
    x='State',
    y='Sales',
    title='Top 10 estados por ventas totales',
    labels={'Sales': 'Ventas (USD)', 'State': 'Estado'},
    color='Sales',
    color_continuous_scale='Blues'
)

fig7.update_layout(template='plotly_white', xaxis_tickangle=-45)
fig7.show()

**Interpretación:** Este gráfico identifica los estados con mayor volumen de ventas. California y New York tienden a liderar, lo cual se explica por su alta densidad poblacional y poder adquisitivo. Esta información es útil para focalizar esfuerzos logísticos y de marketing.

---
## 4. Mini Dashboard - Resumen ejecutivo

A continuación se presenta un dashboard consolidado con los indicadores clave del análisis.

In [ ]:
# === MINI DASHBOARD ===

# Crear un dashboard con subplots
fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Ventas por Categoría',
        'Ventas por Región',
        'Evolución Anual de Ventas',
        'Ventas por Segmento de Cliente'
    ),
    specs=[[{'type': 'pie'}, {'type': 'pie'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# --- Panel 1: Ventas por Categoría (Pie) ---
ventas_cat = df.groupby('Category')['Sales'].sum().reset_index()
fig_dash.add_trace(
    go.Pie(
        labels=ventas_cat['Category'],
        values=ventas_cat['Sales'],
        name='Categoría',
        marker_colors=px.colors.qualitative.Set2
    ),
    row=1, col=1
)

# --- Panel 2: Ventas por Región (Pie) ---
ventas_reg = df.groupby('Region')['Sales'].sum().reset_index()
fig_dash.add_trace(
    go.Pie(
        labels=ventas_reg['Region'],
        values=ventas_reg['Sales'],
        name='Región',
        marker_colors=px.colors.qualitative.Pastel
    ),
    row=1, col=2
)

# --- Panel 3: Evolución Anual de Ventas (Bar) ---
ventas_anual = df.groupby('Order Year')['Sales'].sum().reset_index()
fig_dash.add_trace(
    go.Bar(
        x=ventas_anual['Order Year'].astype(str),
        y=ventas_anual['Sales'],
        name='Ventas Anuales',
        marker_color='steelblue'
    ),
    row=2, col=1
)

# --- Panel 4: Ventas por Segmento (Bar) ---
ventas_seg = df.groupby('Segment')['Sales'].sum().reset_index()
fig_dash.add_trace(
    go.Bar(
        x=ventas_seg['Segment'],
        y=ventas_seg['Sales'],
        name='Segmento',
        marker_color='coral'
    ),
    row=2, col=2
)

# Layout del dashboard
fig_dash.update_layout(
    title_text='Dashboard de Ventas - Superstore',
    title_x=0.5,
    height=700,
    showlegend=False,
    template='plotly_white'
)

fig_dash.show()

### Indicadores clave (KPIs)

In [ ]:
# Resumen de KPIs
print("=" * 60)
print("        RESUMEN EJECUTIVO - SUPERSTORE SALES")
print("=" * 60)
print(f"\n📊 Periodo analizado: {df['Order Date'].min().strftime('%d/%m/%Y')} - {df['Order Date'].max().strftime('%d/%m/%Y')}")
print(f"📦 Total de transacciones: {df.shape[0]:,}")
print(f"🧾 Órdenes únicas: {df['Order ID'].nunique():,}")
print(f"👥 Clientes únicos: {df['Customer ID'].nunique():,}")
print(f"💰 Ventas totales: ${df['Sales'].sum():,.2f}")
print(f"💵 Venta promedio por transacción: ${df['Sales'].mean():,.2f}")
print(f"📈 Venta máxima en una transacción: ${df['Sales'].max():,.2f}")
print(f"\n🏆 Categoría líder: {df.groupby('Category')['Sales'].sum().idxmax()}")
print(f"🌎 Región líder: {df.groupby('Region')['Sales'].sum().idxmax()}")
print(f"👤 Segmento líder: {df.groupby('Segment')['Sales'].sum().idxmax()}")
print(f"🏙️  Estado líder: {df.groupby('State')['Sales'].sum().idxmax()}")
print(f"\n🚚 Días promedio de envío: {df['Shipping Days'].mean():.1f} días")
print("=" * 60)

---
## 5. Conclusiones

### Hallazgos principales:

1. **Tendencia de crecimiento:** Las ventas muestran una tendencia al alza a lo largo de los años analizados, con picos estacionales en el último trimestre de cada año.

2. **Categoría líder:** Technology es la categoría que genera mayores ingresos, seguida de Furniture y Office Supplies.

3. **Segmento Consumer dominante:** El segmento de consumidores individuales concentra la mayor proporción de ventas, lo cual sugiere que la estrategia comercial debe seguir enfocándose en este público.

4. **Distribución geográfica:** Las regiones West y East concentran la mayor parte de las ventas. California y New York son los estados con mayor contribución.

5. **Tiempos de envío:** Standard Class es el modo de envío más utilizado pero con tiempos más largos. Same Day cumple con entregas rápidas pero es menos utilizado, posiblemente por su costo.

### Recomendaciones:

- Potenciar las ventas en regiones con menor participación (Central y South) mediante campañas focalizadas.
- Aprovechar la estacionalidad de fin de año para maximizar ingresos con promociones estratégicas.
- Evaluar la posibilidad de expandir opciones de envío rápido para mejorar la experiencia del cliente.
- Invertir en el segmento Corporate que muestra potencial de crecimiento.